## Sanity Check

In [1]:
import pandas as pd

df = pd.read_csv("data/cleaned/streaming_combined_cleaned.csv")

df.head()

,Date,quarter,region,platform,subscribers,revenue,ARPU,subscriber_growth,is_netflix,treated,event_time,DDD
0,2019-03-30,2019Q1,Global,Disney+,28,0.9,10.7,NaN,0,0,NaN,0
1,2019-06-29,2019Q2,Global,Disney+,30,1.0,11.1,2.0,0,0,NaN,0
2,2019-09-28,2019Q3,Global,Disney+,32,1.1,11.5,2.0,0,0,NaN,0
3,2019-12-28,2019Q4,Global,Disney+,35,1.2,11.7,3.0,0,0,NaN,0
4,2020-03-28,2020Q1,Global,Disney+,50,1.6,12.8,15.0,0,0,NaN,0


In [2]:
print(df.columns)
print(df["platform"].unique())
print(df.groupby("platform")["region"].unique())
print(df["quarter"].unique()[:10])

Index(['Date', 'quarter', 'region', 'platform', 'subscribers', 'revenue',
       'ARPU', 'subscriber_growth', 'is_netflix', 'treated', 'event_time',
       'DDD'],
      dtype='object')
['Disney+' 'Netflix']
platform
Disney+                     [Global]
Netflix    [APAC, EMEA, LATAM, UCAN]
Name: region, dtype: object
['2019Q1' '2019Q2' '2019Q3' '2019Q4' '2020Q1' '2020Q2' '2020Q3' '2020Q4'
 '2021Q1' '2021Q2']


### Step 1: Treated Value Verification

In [ ]:
# Policy timing: 2022 Q3
df[df["region"] == "LATAM"][["quarter", "treated"]]

,quarter,treated
66,2019Q1,0
67,2019Q2,0
68,2019Q3,0
69,2019Q4,0
70,2020Q1,0
71,2020Q2,0
72,2020Q3,0
73,2020Q4,0
74,2021Q1,0
75,2021Q2,0


In [ ]:
# Policy timing: 2023 Q1
df[df["region"] == "EMEA"][["quarter", "treated"]]

,quarter,treated
45,2019Q1,0
46,2019Q2,0
47,2019Q3,0
48,2019Q4,0
49,2020Q1,0
50,2020Q2,0
51,2020Q3,0
52,2020Q4,0
53,2021Q1,0
54,2021Q2,0


In [ ]:
# Policy timing: 2023 Q2
df[df["region"] == "UCAN"][["quarter", "treated"]]

,quarter,treated
87,2019Q1,0
88,2019Q2,0
89,2019Q3,0
90,2019Q4,0
91,2020Q1,0
92,2020Q2,0
93,2020Q3,0
94,2020Q4,0
95,2021Q1,0
96,2021Q2,0


In [ ]:
# Policy timing: 2023 Q2
df[df["region"] == "APAC"][["quarter", "treated"]]

,quarter,treated
24,2019Q1,0
25,2019Q2,0
26,2019Q3,0
27,2019Q4,0
28,2020Q1,0
29,2020Q2,0
30,2020Q3,0
31,2020Q4,0
32,2021Q1,0
33,2021Q2,0


The treatment indicator correctly switches from 0 to 1 following the first full quarter of implementation in each region.

### Step 2: Platform Indicator Construction

In [7]:
df["is_netflix"] = (df["platform"] == "Netflix").astype(int)

In [ ]:
# is_netflix value check
df[["platform", "is_netflix"]].drop_duplicates()

,platform,is_netflix
0,Disney+,0
24,Netflix,1


### Step 3: Triple-Difference Variable Construction

In [9]:
df["DDD"] = df["treated"] * df["is_netflix"]

In [ ]:
df[["platform", "treated", "is_netflix", "DDD"]].sample(20)

,platform,treated,is_netflix,DDD
44,Netflix,1,1,1
38,Netflix,0,1,0
83,Netflix,1,1,1
13,Disney+,0,0,0
8,Disney+,0,0,0
0,Disney+,0,0,0
105,Netflix,1,1,1
79,Netflix,0,1,0
10,Disney+,0,0,0
29,Netflix,0,1,0


We define the triple-difference interaction term as the product of the treatment indicator and the Netflix platform indicator, capturing the differential effect of the policy on Netflix relative to other platforms.

### Step 4: Policy Control Variables

In [12]:
df["AdTier"] = (df["quarter"] >= "2022Q4").astype(int)
df["PriceIncrease"] = (df["quarter"] >= "2023Q1").astype(int)

In [ ]:
df[["quarter", "AdTier", "PriceIncrease"]].drop_duplicates().sort_values("quarter")

,quarter,AdTier,PriceIncrease
0,2019Q1,0,0
1,2019Q2,0,0
2,2019Q3,0,0
3,2019Q4,0,0
4,2020Q1,0,0
5,2020Q2,0,0
6,2020Q3,0,0
7,2020Q4,0,0
8,2021Q1,0,0
9,2021Q2,0,0


We include policy control variables to account for concurrent platform changes. Specifically, we control for the introduction of the ad-supported tier and price adjustments, which may independently affect subscriber growth and revenue.

### Step 5: Event Time Validation

In [ ]:
# policy quarter = 0
# before policy quarter < 0
# after policy quarter > 0
df[df["region"] == "LATAM"][["quarter", "event_time"]]

,quarter,event_time
66,2019Q1,-14.0
67,2019Q2,-13.0
68,2019Q3,-12.0
69,2019Q4,-11.0
70,2020Q1,-10.0
71,2020Q2,-9.0
72,2020Q3,-8.0
73,2020Q4,-7.0
74,2021Q1,-6.0
75,2021Q2,-5.0


In [16]:
# Final Dataset
df.to_csv("output/analysis_ready_panel.csv", index=False)